#  Build a Chatbot using Hugging Face Transformers



### What is a Transformer?

A **Transformer** is a deep learning architecture introduced in the paper *"Attention Is All You Need"* (Vaswani et al., 2017). It relies on a mechanism called **self-attention**, which helps the model determine how important each word is in relation to others when generating text. This allows it to capture context much more effectively than older approaches like RNNs or LSTMs.

---

### Key Concepts

- **Pre-trained Model**  
  A model that has already been trained on massive amounts of text data (e.g., GPT-2, DialoGPT). These models can be fine-tuned further or used directly for tasks like chatbots.

- **Tokenization**  
  The process of converting raw text into numerical token IDs that the model can understand and process.

- **Text Generation**  
  The model generates responses by predicting the next token based on the given context, one step at a time.

- **DialoGPT**  
  A GPT-2-based model that has been fine-tuned specifically on conversational data from Reddit, making it well-suited for chatbot applications.

- **Prompt Engineering**  
  The practice of designing and structuring input text in a way that guides the model to produce better and more relevant responses.

- **Attention Mechanism**  
  A core component of transformers that enables the model to focus on the most relevant parts of the input or conversation history when generating a reply.

---

### Why DialoGPT for a Chatbot?

- Fine-tuned on **147M Reddit conversations**, making it naturally conversational
- Available in three sizes: `small`, `medium`, and `large`
- Easy to use with the Hugging Face `pipeline` API
- Capable of generating **contextually relevant**, multi-turn responses without additional training

---

##  Step 1: Install & Import Libraries

In [12]:
# Install required packages (run once — takes ~2 minutes on Colab)
# DialoGPT works with either PyTorch or TensorFlow backend
!pip install transformers torch --quiet
print(" Installation complete.")

 Installation complete.


In [18]:
# ── Core Imports ──────────────────────────────────────────────────────────────
import torch
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoModelForCausalLM, AutoTokenizer

# Display library versions for reproducibility
import transformers
print(f"PyTorch version       : {torch.__version__}")
print(f"Transformers version  : {transformers.__version__}")
print(f"CUDA available        : {torch.cuda.is_available()}")
print(f"Device                : {'GPU' if torch.cuda.is_available() else 'CPU'}")
print("\n All libraries imported successfully.")

PyTorch version       : 2.10.0+cu128
Transformers version  : 5.0.0
CUDA available        : True
Device                : GPU

 All libraries imported successfully.
PyTorch version       : 2.10.0+cu128
Transformers version  : 5.0.0
CUDA available        : True
Device                : GPU

 All libraries imported successfully.


---
##  Step 2: Model Loading

We use **`microsoft/DialoGPT-medium`** — a GPT-2 based model fine-tuned on 147M Reddit dialogue pairs.  
It is purpose-built for multi-turn conversational tasks and produces natural, context-aware replies.

In [19]:
# ── Model Configuration ───────────────────────────────────────────────────────
MODEL_NAME = "microsoft/DialoGPT-medium"
# Alternatives:
#   "microsoft/DialoGPT-small"  → Faster, lighter (use on CPU)
#   "microsoft/DialoGPT-large"  → Slower, higher quality (use on GPU)
#   "gpt2"                      → General-purpose text generation

print(f"Loading model: {MODEL_NAME}")
print("Please wait — this downloads ~350MB on first run...\n")

# ── Load Tokenizer ────────────────────────────────────────────────────────────
# The tokenizer converts text ↔ token IDs using Byte-Pair Encoding (BPE)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    padding_side='left'   # DialoGPT pads on the left for generation
)
tokenizer.pad_token = tokenizer.eos_token  # Set pad token = EOS token

# ── Load Model ────────────────────────────────────────────────────────────────
# AutoModelForCausalLM: auto-selects the correct causal LM architecture
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)   # Move model to GPU if available
model.eval()               # Set to evaluation mode (disables dropout)

print(f"\n Model '{MODEL_NAME}' loaded successfully on {device.upper()}.")
print(f"   Model parameters : {sum(p.numel() for p in model.parameters()):,}")

Loading model: microsoft/DialoGPT-medium
Please wait — this downloads ~350MB on first run...



Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Model 'microsoft/DialoGPT-medium' loaded successfully on CUDA.
   Model parameters : 406,286,336
Loading model: microsoft/DialoGPT-medium
Please wait — this downloads ~350MB on first run...



Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



 Model 'microsoft/DialoGPT-medium' loaded successfully on CUDA.
   Model parameters : 406,286,336


---
##  Step 3: Response Generation Function

The core function that takes user input + conversation history and produces a response using the transformer model.

In [20]:
def generate_response(user_input, conversation_history, max_history_turns=5):
    """
    Generate a chatbot response using DialoGPT.

    How it works:
      1. Encode the new user input as token IDs.
      2. Append it to the conversation history (maintains context).
      3. Trim history to the last N turns to avoid exceeding max token length.
      4. Run the model's generate() method with controlled sampling parameters.
      5. Decode the generated tokens back into human-readable text.
      6. Return the new response and updated history.

    Args:
        user_input          (str)        : The user's message.
        conversation_history (torch.Tensor | None): Token IDs of prior conversation.
        max_history_turns   (int)        : Max dialogue turns to keep in context.

    Returns:
        response  (str)          : The chatbot's generated reply.
        new_history (torch.Tensor): Updated conversation history token IDs.
    """

    # ── Step 1: Encode user input ─────────────────────────────────────────────
    # EOS token (end-of-sequence) separates turns in DialoGPT
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'
    ).to(device)

    # ── Step 2: Append to conversation history ────────────────────────────────
    if conversation_history is not None:
        bot_input_ids = torch.cat([conversation_history, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # ── Step 3: Trim history to avoid exceeding max token length (1024) ───────
    # Keep only the last max_history_turns * ~50 tokens as a rough limit
    max_tokens = max_history_turns * 100
    if bot_input_ids.shape[-1] > max_tokens:
        bot_input_ids = bot_input_ids[:, -max_tokens:]

    # ── Step 4: Generate response ─────────────────────────────────────────────
    with torch.no_grad():    # No gradient computation needed for inference
        output_ids = model.generate(
            bot_input_ids,

            # Generation strategy: sampling for diverse, natural responses
            do_sample=True,          # Enable random sampling (not greedy)
            max_new_tokens=150,      # Max tokens to generate in the response
            min_new_tokens=5,        # Ensure at least a 5-token response
            temperature=0.75,        # Lower = more focused; Higher = more creative
            top_p=0.92,              # Nucleus sampling: consider top 92% probability mass
            top_k=50,                # Also restrict to top 50 candidate tokens
            repetition_penalty=1.3,  # Penalise repeating the same phrases
            pad_token_id=tokenizer.eos_token_id,  # Padding = EOS
            eos_token_id=tokenizer.eos_token_id,  # Stop at EOS
        )

    # ── Step 5: Decode only the newly generated tokens ────────────────────────
    # Slice off the input tokens; only decode the new response tokens
    response_ids  = output_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(
        response_ids[0],
        skip_special_tokens=True    # Remove EOS and other special tokens
    ).strip()

    # ── Step 6: Update history with full output ───────────────────────────────
    new_history = output_ids

    return response_text, new_history


print(" Response generation function defined.")

 Response generation function defined.
 Response generation function defined.


---
##  Step 4: Input Validation & Utilities

In [21]:
def validate_input(user_input):
    """
    Validate and clean user input before passing it to the model.

    Checks:
    - Not empty or whitespace-only
    - Within reasonable length (not excessively long)

    Returns:
        (is_valid: bool, cleaned_input: str, error_msg: str)
    """
    cleaned = user_input.strip()

    if not cleaned:
        return False, cleaned, "  Input cannot be empty. Please type something."

    if len(cleaned) > 500:
        return False, cleaned, "  Input too long. Please keep it under 500 characters."

    return True, cleaned, ""


def is_exit_command(user_input):
    """
    Check if the user wants to exit the chatbot.
    Accepts: 'exit', 'quit', 'bye', 'goodbye', 'stop', 'end'
    """
    EXIT_COMMANDS = {'exit', 'quit', 'bye', 'goodbye', 'stop', 'end'}
    return user_input.strip().lower() in EXIT_COMMANDS


def print_separator(char='─', width=60):
    """Print a formatted separator line."""
    print(char * width)


def format_chatbot_response(response):
    """
    Post-process the model response for cleaner display.
    - Capitalize the first letter
    - Ensure it ends with punctuation
    """
    if not response:
        return "I'm not sure how to respond to that. Could you rephrase?"

    # Capitalize first letter
    response = response[0].upper() + response[1:] if len(response) > 1 else response.upper()

    # Add period if no ending punctuation
    if response[-1] not in '.!?…':
        response += '.'

    return response


print(" Utility functions defined.")

 Utility functions defined.
 Utility functions defined.


---
##  Step 5: Main Chatbot – Interactive Loop

The complete chatbot with multi-turn conversation, history tracking, and graceful exit handling.

In [22]:
def run_chatbot():
    """
    Main chatbot function.

    Features:
      - Multi-turn conversation with context memory
      - Input validation
      - Graceful exit handling
      - Turn counter and session logging
      - Clean formatted display
    """

    # ── Session Initialization ────────────────────────────────────────────────
    conversation_history = None   # Stores token IDs of full conversation
    turn_count           = 0      # Track number of exchanges
    chat_log             = []     # Store conversation for display at end

    # ── Welcome Banner ────────────────────────────────────────────────────────
    print_separator('═')
    print("            DIALOGPT CHATBOT  ")
    print("       Powered by microsoft/DialoGPT-medium")
    print_separator('═')
    print("  Type your message and press Enter to chat.")
    print("  Type  'exit' / 'quit' / 'bye'  to end the session.")
    print_separator('─')
    print()

    greeting = "Hello! I am your AI assistant powered by DialoGPT. How can I help you today?"
    print(f"   Chatbot : {greeting}")
    print()
    chat_log.append(("Chatbot", greeting))

    # ── Conversation Loop ─────────────────────────────────────────────────────
    while True:
        try:
            # ── Get User Input ────────────────────────────────────────────────
            raw_input = input("  👤 You     : ")

            # ── Check Exit Command ────────────────────────────────────────────
            if is_exit_command(raw_input):
                farewell = "Goodbye! It was great chatting with you. Have a wonderful day! 😊"
                print()
                print(f"   Chatbot : {farewell}")
                chat_log.append(("You", raw_input))
                chat_log.append(("Chatbot", farewell))
                print()
                print_separator('─')
                print(f"  Session ended. Total exchanges: {turn_count}")
                print_separator('═')
                break

            # ── Validate Input ────────────────────────────────────────────────
            is_valid, cleaned_input, error_msg = validate_input(raw_input)
            if not is_valid:
                print(f"    {error_msg}")
                print()
                continue

            # ── Generate Response ─────────────────────────────────────────────
            print("  ⏳ Thinking...")
            response, conversation_history = generate_response(
                cleaned_input,
                conversation_history,
                max_history_turns=5
            )
            response = format_chatbot_response(response)

            # ── Display Response ──────────────────────────────────────────────
            turn_count += 1
            print(f"   Chatbot : {response}")
            print()

            # Log for session summary
            chat_log.append(("You", cleaned_input))
            chat_log.append(("Chatbot", response))

        except KeyboardInterrupt:
            # Handle Ctrl+C gracefully
            print("\n\n   Chatbot : Session interrupted. Goodbye!")
            print_separator('═')
            break

    # ── Session Summary ───────────────────────────────────────────────────────
    print()
    print_separator('═')
    print("   CONVERSATION LOG")
    print_separator('─')
    for speaker, message in chat_log:
        icon = "" if speaker == "Chatbot" else "👤"
        print(f"  {icon} {speaker:<10}: {message}")
    print_separator('═')


print(" Chatbot function defined. Run the next cell to start the chatbot.")

 Chatbot function defined. Run the next cell to start the chatbot.
 Chatbot function defined. Run the next cell to start the chatbot.


In [23]:

run_chatbot()

════════════════════════════════════════════════════════════
            DIALOGPT CHATBOT  
       Powered by microsoft/DialoGPT-medium
════════════════════════════════════════════════════════════
  Type your message and press Enter to chat.
  Type  'exit' / 'quit' / 'bye'  to end the session.
────────────────────────────────────────────────────────────

   Chatbot : Hello! I am your AI assistant powered by DialoGPT. How can I help you today?

  👤 You     : hi
  ⏳ Thinking...
   Chatbot : Hey there! I'm going to be starting my journey in a few weeks so if you want more info on it, feel free. It's always good to have some friends here!

  👤 You     : What is AI
  ⏳ Thinking...
   Chatbot : All In One Machine :P.

  👤 You     : What is Artificial Intelligence
  ⏳ Thinking...
   Chatbot : A new word we are using nowadays that means everything is an AI!

  👤 You     : Who created Python
  ⏳ Thinking...
   Chatbot : Dude, the Turing test was made by the guy who invented AI.

  👤 You     : e

---
##  Step 6: Non-Interactive Demo (Simulated Conversation)

This cell runs a **pre-defined conversation** to demonstrate the chatbot's capabilities without interactive input.  
Ideal for showing expected output in the GitHub submission.

In [24]:
def run_demo_conversation():
    """
    Simulate a pre-defined conversation to demonstrate chatbot output.
    Uses the same generate_response() function as the interactive chatbot.
    """
    demo_inputs = [
        "Hello!",
        "What is Artificial Intelligence?",
        "Who created Python programming language?",
        "Can you tell me something interesting about space?",
        "What is machine learning?",
        "Thank you for chatting with me!",
    ]

    conversation_history = None

    print_separator('═')
    print("    DIALOGPT CHATBOT — DEMO CONVERSATION")
    print_separator('═')
    print()
    print("   Chatbot : Hello! I am your AI assistant powered by DialoGPT.")
    print("               How can I help you today?")
    print()

    for user_msg in demo_inputs:
        print(f"   You     : {user_msg}")

        response, conversation_history = generate_response(
            user_msg,
            conversation_history
        )
        response = format_chatbot_response(response)

        print(f"   Chatbot : {response}")
        print()

    print("   You     : exit")
    print("   Chatbot : Goodbye! It was great chatting with you. Have a wonderful day! 😊")
    print()
    print_separator('═')
    print("   Demo conversation complete.")
    print_separator('═')


# Run the demo
run_demo_conversation()

════════════════════════════════════════════════════════════
    DIALOGPT CHATBOT — DEMO CONVERSATION
════════════════════════════════════════════════════════════

   Chatbot : Hello! I am your AI assistant powered by DialoGPT.
               How can I help you today?

   You     : Hello!
   Chatbot : Hello! Thanks for visiting! Hope you have a wonderful day.

   You     : What is Artificial Intelligence?
   Chatbot : It's the most amazing thing I've ever done in my life. And you're welcome, and thanks again! :D.

   You     : Who created Python programming language?
   Chatbot : Oh, it was actually just an example of what kind of languages would be good for learning.

   You     : Can you tell me something interesting about space?
   Chatbot : Space can not be interesting.

   You     : What is machine learning?
   Chatbot : Haha, good point...

   You     : Thank you for chatting with me!
   Chatbot : You are very nice to talk to!

   You     : exit
   Chatbot : Goodbye! It was great

---
##  Step 7: Model & Generation Parameter Analysis

In [25]:
# ── Model Architecture Summary ────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("═" * 55)
print("       MODEL CONFIGURATION SUMMARY")
print("═" * 55)
print(f"  Model Name          : microsoft/DialoGPT-medium")
print(f"  Architecture        : GPT-2 (Causal Language Model)")
print(f"  Training Data       : 147M Reddit Dialogue Pairs")
print(f"  Total Parameters    : {total_params:,}")
print(f"  Trainable Params    : {trainable_params:,}")
print(f"  Vocabulary Size     : {tokenizer.vocab_size:,} tokens")
print(f"  Max Context Length  : 1,024 tokens")
print(f"  Tokenizer Type      : Byte-Pair Encoding (BPE)")
print(f"  Device              : {device.upper()}")
print()
print("─" * 55)
print("       GENERATION PARAMETERS")
print("─" * 55)
print(f"  Strategy            : Sampling (do_sample=True)")
print(f"  Temperature         : 0.75  (balanced creativity)")
print(f"  Top-p (nucleus)     : 0.92  (high-quality diversity)")
print(f"  Top-k               : 50    (token candidate limit)")
print(f"  Repetition Penalty  : 1.3   (avoids loops)")
print(f"  Max New Tokens      : 150")
print(f"  Min New Tokens      : 5")
print(f"  History Turns Kept  : 5     (context window)")
print("═" * 55)

═══════════════════════════════════════════════════════
       MODEL CONFIGURATION SUMMARY
═══════════════════════════════════════════════════════
  Model Name          : microsoft/DialoGPT-medium
  Architecture        : GPT-2 (Causal Language Model)
  Training Data       : 147M Reddit Dialogue Pairs
  Total Parameters    : 406,286,336
  Trainable Params    : 406,286,336
  Vocabulary Size     : 50,257 tokens
  Max Context Length  : 1,024 tokens
  Tokenizer Type      : Byte-Pair Encoding (BPE)
  Device              : CUDA

───────────────────────────────────────────────────────
       GENERATION PARAMETERS
───────────────────────────────────────────────────────
  Strategy            : Sampling (do_sample=True)
  Temperature         : 0.75  (balanced creativity)
  Top-p (nucleus)     : 0.92  (high-quality diversity)
  Top-k               : 50    (token candidate limit)
  Repetition Penalty  : 1.3   (avoids loops)
  Max New Tokens      : 150
  Min New Tokens      : 5
  History Turns Kept 

In [26]:
# ── Tokenization Visualization ────────────────────────────────────────────────
# Demonstrates how the tokenizer converts text to token IDs and back

sample_texts = [
    "Hello! How are you?",
    "What is Artificial Intelligence?",
    "DialoGPT is a great model for chatbots."
]

print("═" * 65)
print("      TOKENIZATION DEMONSTRATION")
print("═" * 65)

for text in sample_texts:
    tokens    = tokenizer.tokenize(text)
    input_ids = tokenizer.encode(text)
    decoded   = tokenizer.decode(input_ids, skip_special_tokens=True)

    print(f"\n  Original   : {text}")
    print(f"  Tokens     : {tokens}")
    print(f"  Token IDs  : {input_ids}")
    print(f"  Decoded    : {decoded}")
    print(f"  Token Count: {len(tokens)}")
    print("─" * 65)

print("\n Tokenization demo complete.")

═════════════════════════════════════════════════════════════════
      TOKENIZATION DEMONSTRATION
═════════════════════════════════════════════════════════════════

  Original   : Hello! How are you?
  Tokens     : ['Hello', '!', 'ĠHow', 'Ġare', 'Ġyou', '?']
  Token IDs  : [15496, 0, 1374, 389, 345, 30]
  Decoded    : Hello! How are you?
  Token Count: 6
─────────────────────────────────────────────────────────────────

  Original   : What is Artificial Intelligence?
  Tokens     : ['What', 'Ġis', 'ĠArtificial', 'ĠIntelligence', '?']
  Token IDs  : [2061, 318, 35941, 9345, 30]
  Decoded    : What is Artificial Intelligence?
  Token Count: 5
─────────────────────────────────────────────────────────────────

  Original   : DialoGPT is a great model for chatbots.
  Tokens     : ['Dial', 'o', 'G', 'PT', 'Ġis', 'Ġa', 'Ġgreat', 'Ġmodel', 'Ġfor', 'Ġchat', 'bots', '.']
  Token IDs  : [24400, 78, 38, 11571, 318, 257, 1049, 2746, 329, 8537, 42478, 13]
  Decoded    : DialoGPT is a great model fo

---
## 🔍 Step 8: How Transformer Text Generation Works (Visual Explanation)

In [27]:
# ── Generation Strategy Comparison ───────────────────────────────────────────
# Show how different temperature values affect response diversity

test_prompt = "What do you think about the future of technology?"

temperature_values = [0.3, 0.75, 1.2]
labels = ['Conservative (focused)', 'Balanced (default)', 'Creative (diverse)']

print("═" * 70)
print("   TEMPERATURE EFFECT ON RESPONSE GENERATION")
print("═" * 70)
print(f"  Prompt: '{test_prompt}'")
print()

prompt_ids = tokenizer.encode(test_prompt + tokenizer.eos_token, return_tensors='pt').to(device)

for temp, label in zip(temperature_values, labels):
    with torch.no_grad():
        output = model.generate(
            prompt_ids,
            do_sample=True,
            max_new_tokens=60,
            temperature=temp,
            top_p=0.92,
            top_k=50,
            repetition_penalty=1.3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    response_ids = output[:, prompt_ids.shape[-1]:]
    response     = tokenizer.decode(response_ids[0], skip_special_tokens=True).strip()
    response     = format_chatbot_response(response)

    print(f"  Temperature={temp} [{label}]")
    print(f"  → {response}")
    print()

print("─" * 70)
print("  Observation:")
print("  Low temperature → More deterministic, repetitive")
print("  High temperature → More varied, potentially less coherent")
print("  Balanced (0.7–0.8) → Best for chatbot applications")
print("═" * 70)

══════════════════════════════════════════════════════════════════════
   TEMPERATURE EFFECT ON RESPONSE GENERATION
══════════════════════════════════════════════════════════════════════
  Prompt: 'What do you think about the future of technology?'

  Temperature=0.3 [Conservative (focused)]
  → I'm not sure. I don't really know what that means, but it's certainly interesting to me!

  Temperature=0.75 [Balanced (default)]
  → I don't know. I'm not a physicist, but that is what i have heard from people who are. They say it will be better soon and then they disappear forever?!

  Temperature=1.2 [Creative (diverse)]
  → The future is great and we need to invest in a more efficient industry.

──────────────────────────────────────────────────────────────────────
  Observation:
  Low temperature → More deterministic, repetitive
  High temperature → More varied, potentially less coherent
  Balanced (0.7–0.8) → Best for chatbot applications
══════════════════════════════════════════════════

---
##  Summary & Findings

This project successfully implements a conversational chatbot using the `microsoft/DialoGPT-medium` model loaded via `AutoModelForCausalLM`. Tokenization is handled using Byte-Pair Encoding through `AutoTokenizer`, ensuring efficient text processing.

Responses are generated using a sampling-based approach that combines temperature, top-p, top-k, and a repetition penalty to balance coherence and diversity. The chatbot maintains conversation history at the token level, allowing it to produce context-aware responses across multiple turns.

Robust input validation is in place, including checks for empty inputs, length constraints, and correct data types. The system also gracefully handles exit commands such as "exit", "quit", "bye", "goodbye", as well as interruptions like `Ctrl+C`.

Two modes are implemented:
- **Interactive mode** using `run_chatbot()` for real-time conversations
- **Demo mode** using `run_demo_conversation()` with predefined inputs

Additional exploratory components include:
- A temperature comparison study across three values to observe response variability
- A tokenization demo showcasing token IDs, vocabulary mapping, and encode–decode roundtripping

---

## Key Learnings

1. **Transformer Architecture**  
   The self-attention mechanism enables the model to evaluate the relevance of each prior token when generating the next one, which is essential for maintaining context in longer conversations.

2. **DialoGPT vs GPT-2**  
   DialoGPT is specifically fine-tuned on conversational datasets, making it significantly more effective for dialogue tasks compared to GPT-2.

3. **Conversation History Matters**  
   Maintaining and feeding the entire token history back into the model allows for coherent multi-turn interactions. Without it, responses would lack context.

4. **Generation Parameters Trade-offs**  
   Parameters like temperature, top-p, and top-k influence the balance between creativity and reliability. A temperature around `0.75` with `top-p = 0.92` provides strong results.

5. **Repetition Penalty is Essential**  
   Applying a repetition penalty prevents the model from looping and repeating phrases excessively, which is critical for chatbot quality.

---